In [1]:
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
unit_dispatches_dataframe = pd.read_excel("unit_dispatches_last_30_days.xlsx", index_col=0)
incidents_dataframe = pd.read_excel("unit_dispatches_last_30_days.xlsx", index_col=0)

In [3]:
unit_dispatches_dataframe

,unit,dispatched,arrived,in_service,is_in_charge,incident_number
0,A10,12:55,NaN,13:07,True,F260019908
1,E27,12:55,13:05,13:38,False,F260019908
2,E33,12:55,13:03,13:19,False,F260019908
3,H98,12:59,NaN,13:07,False,F260019908
4,A14,12:51,12:56,13:22,True,F260019906
...,...,...,...,...,...,...
14027,R1,00:36,00:43,00:51,False,F260005146
14028,E31,00:28,00:33,00:51,True,F260005144
14029,E25,00:18,00:20,00:30,True,F260005141
14030,E25,00:01,00:09,00:18,True,F260005139


In [4]:
unit_dispatches_dataframe['unit'].value_counts()[:10]

unit
A2     486
A25    483
A5     461
A10    460
E2     385
E25    374
M1     367
E17    345
M10    334
E10    327
Name: count, dtype: int64

I want to add two new columns to `unit_dispatches_dataframe`. `time_in_transit` should be the difference between `dispatched` and `arrived`, `NaN` if `arrived` is `NaN`. `time_on_site` should be the difference between `arrived` and `in_service`, `NaN` if `arrived` is `NaN`.

In [5]:
def calculate_time_delta(time1, time2):
    if pd.isna(time1) or pd.isna(time2):
        return float('nan')
    time1_hours, time1_minutes = [int(value) for value in time1.split(":")]
    time1_total_minutes = time1_minutes + time1_hours * 60
    time2_hours, time2_minutes = [int(value) for value in time2.split(":")]
    if time2_hours == 0 and time1_hours != 0: 
        # If time1 is 23:50 and time2 is 1:05, this will fail, need a more general solution
        time2_hours = 24
    time2_total_minutes = time2_minutes + time2_hours * 60
    return abs(time1_total_minutes - time2_total_minutes)

In [6]:
unit_dispatches_dataframe['time_in_transit'] = unit_dispatches_dataframe.apply(
    lambda row : calculate_time_delta(
        row['dispatched'],
        row['arrived']
    ), axis=1
)

In [7]:
unit_dispatches_dataframe['time_on_site'] = unit_dispatches_dataframe.apply(
    lambda row : calculate_time_delta(
        row['arrived'],
        row['in_service']
    ), axis=1
)

In [8]:
unit_dispatches_dataframe['time_assigned'] = unit_dispatches_dataframe.apply(
    lambda row : calculate_time_delta(
        row['dispatched'],
        row['in_service']
    ), axis=1
)

In [9]:
pd.set_option('display.max_rows', None)

unit_dispatches_dataframe

,unit,dispatched,arrived,in_service,is_in_charge,incident_number,time_in_transit,time_on_site,time_assigned
0,A10,12:55,NaN,13:07,True,F260019908,NaN,NaN,12.0
1,E27,12:55,13:05,13:38,False,F260019908,10.0,33.0,43.0
2,E33,12:55,13:03,13:19,False,F260019908,8.0,16.0,24.0
3,H98,12:59,NaN,13:07,False,F260019908,NaN,NaN,8.0
4,A14,12:51,12:56,13:22,True,F260019906,5.0,26.0,31.0
5,H2,12:42,13:01,13:48,True,F260019902,19.0,47.0,66.0
6,A14,12:41,NaN,12:45,True,F260019900,NaN,NaN,4.0
7,H98,12:44,NaN,12:45,False,F260019900,NaN,NaN,1.0
8,E25,12:40,12:51,12:52,True,F260019899,11.0,1.0,12.0
9,E40,12:40,12:46,12:57,True,F260019897,6.0,11.0,17.0


In [10]:
unit_dispatches_dataframe[unit_dispatches_dataframe["unit"] == "MAR5"]["time_assigned"].mean()

np.float64(39.31707317073171)

In [11]:
pd.Series([5, 4, float('nan')]).mean()

np.float64(4.5)

In [12]:
unit_list = sorted(unit_dispatches_dataframe["unit"].unique())

In [13]:
unit_dataframe = pd.DataFrame(
    columns=[
        "unit", 
        "number_of_incidents", 
        "leadership_rate", 
        "average_time_in_transit", "total_time_in_transit", 
        "average_time_on_site", "total_time_on_site", 
        "average_time_assigned", "total_time_assigned"
    ]
)

In [14]:
filtered_unit_dispatches_dataframe = unit_dispatches_dataframe[unit_dispatches_dataframe["unit"]=="A17"]

In [15]:
number_of_incidents = len(filtered_unit_dispatches_dataframe)
number_of_incidents

28

In [16]:
in_charge_counts = filtered_unit_dispatches_dataframe["is_in_charge"].value_counts()
in_charge_count = in_charge_counts.get(True) if in_charge_counts.get(True) else 0
leadership_rate = in_charge_count / number_of_incidents
leadership_rate

np.float64(0.8214285714285714)

In [17]:
average_time_in_transit = filtered_unit_dispatches_dataframe["time_in_transit"].mean()
average_time_in_transit

np.float64(5.72)

In [18]:
total_time_in_transit = filtered_unit_dispatches_dataframe["time_in_transit"].sum()
total_time_in_transit

np.float64(143.0)

In [19]:
average_time_on_site = filtered_unit_dispatches_dataframe["time_on_site"].mean()
average_time_on_site

np.float64(18.88)

In [20]:
total_time_on_site = filtered_unit_dispatches_dataframe["time_on_site"].sum()
total_time_on_site

np.float64(472.0)

In [21]:
average_time_assigned = filtered_unit_dispatches_dataframe["time_assigned"].mean()
average_time_assigned

np.float64(22.5)

In [22]:
total_time_assigned = filtered_unit_dispatches_dataframe["time_assigned"].sum()
total_time_assigned

np.float64(630.0)

In [23]:
for unit in unit_list:
    filtered_unit_dispatches_dataframe = unit_dispatches_dataframe[unit_dispatches_dataframe["unit"]==unit]
    number_of_incidents = len(filtered_unit_dispatches_dataframe)
    in_charge_counts = filtered_unit_dispatches_dataframe["is_in_charge"].value_counts()
    in_charge_count = in_charge_counts.get(True) if in_charge_counts.get(True) else 0
    leadership_rate = in_charge_count / number_of_incidents
    average_time_in_transit = filtered_unit_dispatches_dataframe["time_in_transit"].mean()
    total_time_in_transit = filtered_unit_dispatches_dataframe["time_in_transit"].sum()
    average_time_on_site = filtered_unit_dispatches_dataframe["time_on_site"].mean()
    total_time_on_site = filtered_unit_dispatches_dataframe["time_on_site"].sum()
    average_time_assigned = filtered_unit_dispatches_dataframe["time_assigned"].mean()
    total_time_assigned = filtered_unit_dispatches_dataframe["time_assigned"].sum()
    new_row = {
        'unit': unit, 
        'number_of_incidents': number_of_incidents,
        'leadership_rate': leadership_rate,
        'average_time_in_transit': average_time_in_transit,
        'total_time_in_transit': total_time_in_transit,
        'average_time_on_site': average_time_on_site,
        'total_time_on_site': total_time_on_site,
        'average_time_assigned': average_time_assigned,
        'total_time_assigned': total_time_assigned,
    }
    unit_dataframe = pd.concat([unit_dataframe, pd.DataFrame([new_row])], ignore_index=True)

In [24]:
unit_dataframe

,unit,number_of_incidents,leadership_rate,average_time_in_transit,total_time_in_transit,average_time_on_site,total_time_on_site,average_time_assigned,total_time_assigned
0,A10,460,0.791304,4.238663,1776.0,21.406699,8948.0,23.714597,10885.0
1,A14,109,0.688073,6.163636,339.0,18.727273,1030.0,14.926606,1627.0
2,A17,28,0.821429,5.72,143.0,18.88,472.0,22.5,630.0
3,A2,486,0.771605,4.706731,1958.0,17.595181,7302.0,19.523711,9469.0
4,A22,3,1.0,6.0,18.0,31.0,93.0,37.0,111.0
5,A25,483,0.786749,4.816555,2153.0,18.955257,8473.0,22.395445,10817.0
6,A26,2,1.0,5.0,5.0,NaN,0.0,1.0,1.0
7,A31,153,0.745098,5.577465,792.0,20.626761,2929.0,24.633987,3769.0
8,A4,305,0.75082,4.44,1221.0,19.338182,5318.0,21.721311,6625.0
9,A5,461,0.772234,4.949029,2039.0,19.048662,7829.0,22.054348,10145.0


In [25]:
unit_dataframe.to_excel("unit_stats_last_30_days.xlsx")

Leadership rate should never be greater than 1, something is wrong.